# Explore Wikidata staging output

Quick data quality checks on the raw Wikidata extraction before building the next pipeline stage:
- how many records came back
- how many carry a direct IMDb / TMDb ID (Layer 1 matching material)
- whether the same Wikidata item appears more than once (which would need investigating before we trust these counts)

In [16]:
import json
from pathlib import Path
from collections import Counter

STAGING_DIR = Path("../staging_data/wikidata")

# Pick the most recent staging file automatically
latest_file = sorted(STAGING_DIR.glob("wikidata_raw_*.json"))[-1]
print(f"Loading: {latest_file.name}")

with open(latest_file, encoding="utf-8") as f:
    data = json.load(f)

print(f"Total rows: {len(data)}")

Loading: wikidata_raw_20260913T014340Z.json
Total rows: 5371


In [17]:
# Check for duplicate items
item_ids = [row["item"]["value"] for row in data]
unique_items = set(item_ids)

print(f"Total rows:    {len(item_ids)}")
print(f"Unique items:  {len(unique_items)}")
print(f"Difference:    {len(item_ids) - len(unique_items)}")

Total rows:    5371
Unique items:  5210
Difference:    161


In [18]:
# If there are duplicates, look at a few examples and why they duplicated
counts = Counter(item_ids)
dupes = [item for item, c in counts.items() if c > 1]
print(f"{len(dupes)} items appear more than once\n")

for item in dupes[:5]:
    rows = [r for r in data if r["item"]["value"] == item]
    label = rows[0].get("itemLabel", {}).get("value", "(no label)")
    print(f"{item}  —  {label}")
    for r in rows:
        instance_of = r.get("instanceOfLabel", {}).get("value", "?")
        imdb = r.get("imdbId", {}).get("value", "—")
        tmdb = r.get("tmdbId", {}).get("value", "—")
        pub_date = r.get("publicationDate", {}).get("value", "—")
        label_lang = r.get("itemLabel", {}).get("xml:lang", "?")
        print(f"    instance_of={instance_of}  imdb={imdb}  tmdb={tmdb}  pub_date={pub_date}  label_lang={label_lang}")
    print()

122 items appear more than once

http://www.wikidata.org/entity/Q12472132  —  Angkara Murka
    instance_of=film  imdb=—  tmdb=300935  pub_date=1972-01-01T00:00:00Z  label_lang=id
    instance_of=film  imdb=—  tmdb=300935  pub_date=2025-01-01T00:00:00Z  label_lang=id

http://www.wikidata.org/entity/Q7460176  —  Shackled
    instance_of=film  imdb=tt2424752  tmdb=169636  pub_date=2012-01-01T00:00:00Z  label_lang=en
    instance_of=film  imdb=tt2424752  tmdb=169636  pub_date=2013-01-01T00:00:00Z  label_lang=en

http://www.wikidata.org/entity/Q7742388  —  The Intruder
    instance_of=film  imdb=tt0311340  tmdb=81944  pub_date=1985-01-01T00:00:00Z  label_lang=en
    instance_of=film  imdb=tt0311340  tmdb=81944  pub_date=1986-01-01T00:00:00Z  label_lang=en

http://www.wikidata.org/entity/Q7809033  —  Tipu Kanan Tipu Kiri
    instance_of=film  imdb=tt1045893  tmdb=279807  pub_date=2007-01-01T00:00:00Z  label_lang=en
    instance_of=film  imdb=tt1045893  tmdb=279807  pub_date=2008-01-01T00:00

In [19]:
# ID coverage — how much of Layer 1 (exact ID) matching we'll get for free
has_imdb = sum(1 for r in data if "imdbId" in r)
has_tmdb = sum(1 for r in data if "tmdbId" in r)
has_both = sum(1 for r in data if "imdbId" in r and "tmdbId" in r)
has_neither = sum(1 for r in data if "imdbId" not in r and "tmdbId" not in r)

total = len(data)
print(f"Has IMDb ID:      {has_imdb}/{total}  ({has_imdb/total:.1%})")
print(f"Has TMDb ID:      {has_tmdb}/{total}  ({has_tmdb/total:.1%})")
print(f"Has both:         {has_both}/{total}  ({has_both/total:.1%})")
print(f"Has neither:      {has_neither}/{total}  ({has_neither/total:.1%})")

Has IMDb ID:      2725/5371  (50.7%)
Has TMDb ID:      2734/5371  (50.9%)
Has both:         2244/5371  (41.8%)
Has neither:      2156/5371  (40.1%)


In [20]:
# Breakdown by content type (film / TV series / web series / miniseries)
type_counts = Counter(r.get("instanceOfLabel", {}).get("value", "unknown") for r in data)
for content_type, count in type_counts.most_common():
    print(f"{content_type:15s} {count}")

film            3778
television series 1424
web series      145
miniseries      24
